In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import time

base = "/Volumes/workspace/default/m5/"
long = spark.read.parquet(base + "out/sales_long")

cal_schema = StructType([
    StructField("date", StringType()), StructField("wm_yr_wk", IntegerType()),
    StructField("weekday", StringType()), StructField("wday", IntegerType()),
    StructField("month", IntegerType()), StructField("year", IntegerType()),
    StructField("d", StringType()),
    StructField("event_name_1", StringType()), StructField("event_type_1", StringType()),
    StructField("event_name_2", StringType()), StructField("event_type_2", StringType()),
    StructField("snap_CA", IntegerType()), StructField("snap_TX", IntegerType()), StructField("snap_WI", IntegerType()),
])
prices_schema = StructType([
    StructField("store_id", StringType()), StructField("item_id", StringType()),
    StructField("wm_yr_wk", IntegerType()), StructField("sell_price", FloatType()),
])
cal    = spark.read.csv(base + "calendar.csv", header=True, schema=cal_schema) \
              .select("d", F.to_date("date").alias("date"), "wm_yr_wk", "weekday", "event_name_1", "snap_CA", "snap_TX", "snap_WI")
prices = spark.read.csv(base + "sell_prices.csv", header=True, schema=prices_schema)

# ---- Canonical joined table: LEFT join keeps pre-launch days ----------------
# prices.csv only lists weeks an item was on sale; an inner join silently drops
# pre-launch days (12.3M rows, 20.8% — measured in 02). Keep them with
# sell_price = NULL / is_listed = False. Filtering is the analyst's call, not the pipeline's.
joined = (long
    .join(cal, "d", "left")
    .join(prices, ["store_id", "item_id", "wm_yr_wk"], "left")
    .withColumn("is_listed", F.col("sell_price").isNotNull()))

# Sanity check: pre-launch rows should have zero sales.
joined.groupBy("is_listed").agg(F.count("*").alias("rows"), F.sum("sales").alias("total_sales")).orderBy("is_listed").show()

out_joined = base + "out/sales_joined"
t0 = time.time()
joined.write.mode("overwrite").partitionBy("store_id").parquet(out_joined)
print(f"written {joined.count():,} rows — {time.time()-t0:.1f}s")

+---------+--------+-----------+
|is_listed|    rows|total_sales|
+---------+--------+-----------+
|    false|12299413|          0|
|     true|46881677|   66927173|
+---------+--------+-----------+

written 59,181,090 rows — 34.5s


In [0]:
# ---- Window features ---------------------------------------------------------
# Partition by item-store so each series is independent; order by real date.
# rowsBetween(-27, 0) = trailing 28-day window including today.
# This is a wide shuffle (repartition by item-store) + sort within partition —
# the same cost shape as sort-merge join.
w28 = Window.partitionBy("store_id", "item_id").orderBy("date").rowsBetween(-27, 0)
w   = Window.partitionBy("store_id", "item_id").orderBy("date")

feat = (spark.read.parquet(out_joined)
    .withColumn("sales_28d_avg", F.avg("sales").over(w28))
    .withColumn("sales_lag_7",   F.lag("sales", 7).over(w))
    .withColumn("days_since_listed",
        F.when(F.col("is_listed"), F.datediff("date", F.min(F.when(F.col("is_listed"), F.col("date"))).over(w)))))

t0 = time.time()
feat.write.mode("overwrite").partitionBy("store_id").parquet(base + "out/sales_features")
print(f"features written — {time.time()-t0:.1f}s")
feat.filter("item_id = 'HOBBIES_1_001' and store_id = 'CA_1'").select("date","sales","sales_28d_avg","sales_lag_7","days_since_listed").orderBy("date").show(35)

features written — 35.3s
+----------+-----+-------------+-----------+-----------------+
|      date|sales|sales_28d_avg|sales_lag_7|days_since_listed|
+----------+-----+-------------+-----------+-----------------+
|2011-01-29|    0|          0.0|       NULL|             NULL|
|2011-01-30|    0|          0.0|       NULL|             NULL|
|2011-01-31|    0|          0.0|       NULL|             NULL|
|2011-02-01|    0|          0.0|       NULL|             NULL|
|2011-02-02|    0|          0.0|       NULL|             NULL|
|2011-02-03|    0|          0.0|       NULL|             NULL|
|2011-02-04|    0|          0.0|       NULL|             NULL|
|2011-02-05|    0|          0.0|          0|             NULL|
|2011-02-06|    0|          0.0|          0|             NULL|
|2011-02-07|    0|          0.0|          0|             NULL|
|2011-02-08|    0|          0.0|          0|             NULL|
|2011-02-09|    0|          0.0|          0|             NULL|
|2011-02-10|    0|          0.

In [0]:
# days_since_listed should equal 0 on the first listed day, and never be negative
feat.filter("is_listed").agg(
    F.min("days_since_listed").alias("min_dsl"),      # expect 0
    F.max("days_since_listed").alias("max_dsl"),      # ≤ 1940
).show()


+-------+-------+
|min_dsl|max_dsl|
+-------+-------+
|      0|   1940|
+-------+-------+

